In [1]:

import h5py
import numpy as np
from utils import save, load
from library import *
from continuous.process_library_terms import *
import gc
from commons.identify_models import *
import copy
import jax
import jax.numpy as jnp
from jax_cfd.base import boundaries, grids, pressure, finite_differences as fd
import jax_cfd.base as cfd
import jax_cfd.base.grids as grids
import jax_cfd.spectral as spectral
import numpy as np
import xarray 
import xarray as xr
import dataclasses
import time
jax.config.update("jax_enable_x64", True)


In [2]:

f = h5py.File('snapshots/snapshots_s1.h5', 'r')
print(list(f['scales'].keys()))
scales = f['scales']


f = f['tasks']
list(f.keys())



velocity = f['velocity'][800:]
print(velocity.shape)
pressure_import = f['pressure'][800:]
buoyancy = f['buoyancy'][800:]
print(velocity)
print(velocity.shape)

['constant', 'iteration', 'sim_time', 'timestep', 'wall_time', 'write_number', 'x_hash_db097416073d0e2b7376a0ac4c4ada6ab61075c3', 'z_hash_899d48e9709c059e69a87b66f296c00ddc2a045e']
(201, 2, 512, 512)
[[[[ 1.11975321e-03  1.00894879e-02  2.80276071e-02 ... -1.32919886e-02
    -4.78740464e-03 -5.33344350e-04]
   [ 1.14621495e-03  1.03326079e-02  2.86975577e-02 ... -1.24913661e-02
    -4.49348095e-03 -5.00965536e-04]
   [ 1.17399337e-03  1.05759774e-02  2.93643844e-02 ... -1.14511104e-02
    -4.11018408e-03 -4.57272777e-04]
   ...
   [ 1.04498855e-03  9.37793730e-03  2.60538101e-02 ... -1.48066519e-02
    -5.32920032e-03 -5.88898512e-04]
   [ 1.07054128e-03  9.61196299e-03  2.67022242e-02 ... -1.43268487e-02
    -5.15962443e-03 -5.70796814e-04]
   [ 1.09506888e-03  9.84920373e-03  2.73611985e-02 ... -1.38689999e-02
    -4.99649145e-03 -5.54636982e-04]]

  [[ 3.70064157e-07 -9.09288773e-07 -9.72106227e-06 ...  1.06743260e-05
     1.71291418e-06  4.01072559e-07]
   [ 6.75121372e-07 -6.32288

In [3]:
size = 512
inner_steps = 351
outer_steps = 201

Lx =  8
Ly = Lx
# Define the physical dimensions of the simulation.
grid = cfd.grids.Grid((size, size), domain=((0, Lx), (0, Ly)))
Nx = Ny = size
Nt = outer_steps
dy = Lx/size
dx = Ly/size
print(dx)
grid = grids.Grid((size, size), domain=((0, 2 * jnp.pi), (0, 2 * jnp.pi)))
Lt = 0.0002859165868008745 * 175501

dt = Lt/outer_steps*201/1001
print(dt)

Rayleigh = 2e6
Prandtl = 1
kappa = (Rayleigh * Prandtl)**(-1/2)
kappa

0.015625
0.05012851838175851


0.0007071067811865475

In [4]:
P = np.array(pressure_import)
B = np.array(buoyancy)

velocity = np.array(velocity) 
u_trajectory = velocity[:,0,:,:] #128:384
v_trajectory = velocity[:,1,:,:]
P = P[:,:,:]
B = B[:,:,:]
U = np.transpose(u_trajectory, (1,2,0))
V = np.transpose(v_trajectory, (1,2,0))
P = np.transpose(P, (1,2,0))
B = np.transpose(B, (1,2,0))
temp = U
U = V
V = temp

x = scales['x_hash_db097416073d0e2b7376a0ac4c4ada6ab61075c3'][:]#['1.0'][:]    # If your x-basis is named 'x'
y = scales['z_hash_899d48e9709c059e69a87b66f296c00ddc2a045e'][:]#['1.0'][:]    # Likewise for y
t = scales['sim_time'][:]#['1.0'][:]
del(temp)
del(u_trajectory)
del(v_trajectory)
del(velocity)
del(pressure_import)
del(buoyancy)
del(f)
del(scales)

In [5]:
#INTERPOLATION
from scipy.interpolate import Akima1DInterpolator

print(x.shape, y.shape)
print(np.all(np.diff(y) > 0))
X, Y, T = np.meshgrid(x, y, t, indexing='ij')
ys = x
u_lin = Akima1DInterpolator(y, U, axis=1, method='makima')(ys)
print("U_succ")
v_lin = Akima1DInterpolator(y, V, axis=1, method='makima')(ys)
b_lin = Akima1DInterpolator(y, B, axis=1, method='makima')(ys)
p_lin = Akima1DInterpolator(y, P, axis=1, method='makima')(ys)
del(U)
del(V)
del(P)
del(B)

(512,) (512,)
True
U_succ


In [6]:
#%%prun # profiling
#print(P.shape)
uobs = Observable(string='u', rank=1)
pobs = Observable(string='p', rank=0)
bobs = Observable(string='b', rank=0) #?
zobs = Observable(string='z̄', rank=1)
observables = [uobs, bobs]#, pobs, zobs]

u = np.concatenate([v_lin[:, :, :, np.newaxis], u_lin[:, :, :, np.newaxis]], axis=3)
z = np.full(b_lin.shape, 1)
x = np.full(b_lin.shape, 0)
unit_vect = np.concatenate([z[:, :, :, np.newaxis], x[:, :, :, np.newaxis]], axis=3)
data_dict = {'u' : u, 'b' : b_lin}#,'p': p_lin, 'z̄' : unit_vect}
print(u.shape)
# fix random seed
np.random.seed(1)

world_size = np.array(p_lin.shape)
pad = 0

# fix random seed
np.random.seed(1)

dxs = [dx, dy, dt]

# initial setup of dataset
srd = SRDataset(world_size=world_size, data_dict=data_dict, observables=observables, dxs=dxs, 
                irreps=SRDataset.all_rank2_irreps())
                #irreps=(0, 1, 2))

# initialize libraries, domains, and weights
#srd.make_libraries(max_complexity=4, max_observables=3)
srd.make_libraries(max_complexity=3)
for irrep in srd.irreps:
    print(irrep, ":", len(srd.libs[irrep].terms))
 
dom_width = 64 #previously 64 - smaller world size -> smaller domain??
dom_time = 20 #previously 20 (without interpolation)
print(len(srd.libs[srd.irreps[3]].terms))
srd.make_domains(ndomains=150, domain_size=[dom_width, dom_width, dom_time], pad=pad)
#srd.make_domains(ndomains=10, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)

#Nt = 501
#Nx = 256
NLt = Lt*201/1001
Nt = 201
NLx = Lx #/ 2
#NLx = Lx /
print(Lt)
L = NLx/Nx*dom_width
T = NLt/Nt*dom_time
print(NLt)
print(L, T)

srd.set_LT_scale(L, T) #T=1 # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False)

(512, 512, 201, 2)
Rank 0 : 14
Rank 1 : 16
Antisymmetric rank 2 : 5
Symmetric trace-free rank 2 : 8
8
50.17864690014027
10.075832194733461
1.0 1.0025703676351703


In [7]:
from commons.utils import *

lib1 = srd.libs[srd.irreps[1]]
for match in regex_find(lib1.terms, r'∂t u_α'):
    print(match)
dtu = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'u_β · ∂β u_α'):
    print(match)
adv = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'∂α p'):
    print(match)
dp = lib1.Q[:, match[0]]

for match in regex_find(lib1.terms, r'∂β² u_α'):
    print(match)
viscosity = 1e-3*lib1.Q[:, match[0]] #0.1

print(np.linalg.norm(dtu), np.linalg.norm(adv), np.linalg.norm(dp), np.linalg.norm(dtu+dp+adv-viscosity), 
      np.linalg.norm(dp-viscosity), np.linalg.norm(dtu+adv))

(6, ∂t u_α)
(13, u_β · ∂β u_α)
(5, ∂β² u_α)
33885.00133706855 102010.36249213686 102010.36249213686 193107.9473470744 102005.00899926393 93882.83793346798


In [8]:
lib0 = srd.libs[srd.irreps[0]]
for match in regex_find(lib0.terms, r'∂α u_α'):
    print(match)
div = lib0.Q[:, match[0]]

print(np.linalg.norm(div))

(7, ∂α u_α)
0.8255112630972841


In [29]:
from commons.identify_models import *
import copy

libs = srd.libs

reg_opts_list = []
for irrep in srd.irreps:
    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
    scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None)
    init = Initializer(method='combinatorial', start_k=2)
    #init = Initializer(method='combinatorial', start_k=9999)
    #init = Initializer(method='power', start_k=10)
    #res = Residual(residual_type='fixed_column', anchor_col=0)
    res = Residual(residual_type='dominant_balance')
    
    iterator = ModelIterator(max_k=10, backward_forward=True, max_passes=10)
    thres = Threshold(threshold_type='jump', gamma=8, delta=1e-6, n_terms=None) #gamma-delta matter a lot
    #thres = Threshold(threshold_type='information', ic=AIC)
    #thres = Threshold(threshold_type='jump', gamma=1.5, n_terms=3)
    
    opts = {'scaler': scaler, 'initializer': init, 'residual': res,
            'model_iterator': iterator, 'threshold': thres}
    opts['verbose'] = False
    opts['inhomog'] = False
    opts['inhomog_col'] = None
    reg_opts_list.append(opts)

eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify([libs[i] for i in srd.irreps], 
reg_opts_list, threshold=1e-1, experimental=True) #threshold matters
#, max_equations=10)
print(derived_eqs)
print([reg_result.all_lambdas for reg_result in reg_results])

TypeError: interleave_identify() got an unexpected keyword argument 'report_accuracy'

In [10]:
from commons.identify_models import *
import copy
 
libs = srd.libs
 
# Parameter ranges for different aspects of the regression process
threshold_values = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]
initializer_methods = ['combinatorial'] #, 'power'] 
residual_types = ['matrix_relative'] #, 'dominant_balance']
threshold_types = ['information']  #, 'jump'] # Added 'information' type
 
# Iterate through all combinations
for mk in range(5, 40):
    print("###############################################################k = " + str(mk) + "########################################")
    for threshold_loop in threshold_values:
        for init_method in initializer_methods:
            for res_type in residual_types:
                for thres_type in threshold_types:
                    print("-----------------------------------------------------------")
                    print(f"Threshold: {threshold_loop}, Initializer: {init_method}, Residual: {res_type}, Threshold Type: {thres_type}")
                    reg_opts_list = []
                    for irrep in srd.irreps:
                        scaler = Scaler(sub_inds=None, char_sizes=libs[irrep].col_weights, row_norms=None, unit_rows=True, train_fraction=0.75)
                        init = Initializer(method=init_method, start_k=mk)
                        if res_type == "dominant_balance":
                            res = Residual(residual_type='dominant_balance')
                        else:
                            res = Residual(residual_type='matrix_relative')
     
                        iterator = ModelIterator(max_k=mk, backward_forward=False, max_passes=30)
                        if thres_type == "jump":
                            thres = Threshold(threshold_type='jump', gamma=8, delta=1e-6)
                        else:  # 'information'
                            thres = Threshold(threshold_type='information', ic=AIC)
     
                        opts = {
                            'scaler': scaler,
                            'initializer': init,
                            'residual': res,
                            'model_iterator': iterator,
                            'threshold': thres,
                            'verbose': False,
                            'inhomog': False,
                            'inhomog_col': None
                        }
                        reg_opts_list.append(opts)
                    try:
                        eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify(
                            [libs[i] for i in srd.irreps], reg_opts_list, threshold=threshold_loop, experimental=True)
                    except TypeError as e:
                        print(f"Encountered an error: {e}")

###############################################################k = 5########################################
-----------------------------------------------------------
Threshold: 0.01, Initializer: combinatorial, Residual: matrix_relative, Threshold Type: information
--- WORKING ON LIBRARY WITH IRREP Rank 0 AT COMPLEXITY 1 ---
[0.00 s]
Identified model: b = 0 (order 1, train res 3.28e-05, test res 3.28e-05)
--- WORKING ON LIBRARY WITH IRREP Rank 1 AT COMPLEXITY 1 ---
[0.00 s]
Identified model: u_α = 0 (order 1, train res 5.20e-05, test res 5.20e-05)
--- WORKING ON LIBRARY WITH IRREP Antisymmetric rank 2 AT COMPLEXITY 1 ---
--- WORKING ON LIBRARY WITH IRREP Symmetric trace-free rank 2 AT COMPLEXITY 1 ---
--- WORKING ON LIBRARY WITH IRREP Rank 0 AT COMPLEXITY 2 ---
--- WORKING ON LIBRARY WITH IRREP Rank 1 AT COMPLEXITY 2 ---
--- WORKING ON LIBRARY WITH IRREP Antisymmetric rank 2 AT COMPLEXITY 2 ---
--- WORKING ON LIBRARY WITH IRREP Symmetric trace-free rank 2 AT COMPLEXITY 2 ---
--- WOR

In [11]:
''' Threshold: 0.0001, Initializer: power, Residual: matrix_relative, Threshold Type: information
not picking up above 1024
sze 1024 = train res 9.04e-05, test res 7.16e-05
sze 512  = train res 9.33e-05, test res 7.39e-05
sze 256 = train res 9.40e-05, test res 7.45e-05
sze 128 = train res 9.42e-05, test res 7.46e-05
sze 64 = train res 9.43e-05, test res 7.47e-05
sze 32 = train res 9.43e-05, test res 7.47e-05

????Identified model: 0.355 · ∂α b · u_α + ∂t b = 0 (order 3, train res 5.89e-06, test res 1.29e-05)
1.1 e-6 Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00585 · ∂α b · u_α + 0.000168 · ∂α² b + 0.00551 · ∂t b + ∂α u_α = 0 (order 3, train res 1.42e-07, test res 2.34e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00585 · ∂α b · u_α + 0.000168 · ∂α² b + 0.00551 · ∂t b + ∂α u_α = 0 (order 3, train res 1.42e-07, test res 2.34e-07)
Identified model: 0.00131 · b · ∂t b + -0.252 · b · ∂α u_α + 0.00485 · ∂α b · u_α + 0.000182 · ∂α² b + ∂α u_α = 0 (order 3, train res 1.53e-07, test res 2.98e-07)
Identified model: -0.252 · b · ∂α u_α + 0.00476 · ∂α b · u_α + 0.000179 · ∂α² b + 0.00498 · ∂t b + ∂α u_α = 0 (order 3, train res 1.51e-07, test res 2.88e-07)

'''

' Threshold: 0.0001, Initializer: power, Residual: matrix_relative, Threshold Type: information\nnot picking up above 1024\nsze 1024 = train res 9.04e-05, test res 7.16e-05\nsze 512  = train res 9.33e-05, test res 7.39e-05\nsze 256 = train res 9.40e-05, test res 7.45e-05\nsze 128 = train res 9.42e-05, test res 7.46e-05\nsze 64 = train res 9.43e-05, test res 7.47e-05\nsze 32 = train res 9.43e-05, test res 7.47e-05\n\n????Identified model: 0.355 · ∂α b · u_α + ∂t b = 0 (order 3, train res 5.89e-06, test res 1.29e-05)\n1.1 e-6 Identified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)\nIdentified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2.21e-07)\nIdentified model: -0.252 · b · ∂α u_α + 0.00581 · ∂α b · u_α + 0.000173 · ∂α² b + 0.00556 · ∂t b + ∂α u_α = 0 (order 3, train res 1.35e-07, test res 2